In [1]:
import json
import time
import pandas as pd
import numpy as np
from copy import deepcopy as dcopy
import warnings

from sklearn.feature_extraction.text import CountVectorizer

import torch
from torch import nn
import torch.nn.functional as F

/Users/samuele/opt/anaconda3/envs/tuwnlpie/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from running_functions_DL import train, evaluate, training_loop
from utilities_DL import (
    epoch_time,
    prepare_dataloader,
    create_dataloader_iterators,
    create_input,
    prepare_dataloader_with_padding,
    extract_predictions,
    score,
    score_str,
)

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
with open("../../data/tacred/json/train.json", "r") as file:
    data_train = json.load(file)

In [ ]:
with open("../../data/tacred/json/dev.json", "r") as file:
    data_dev = json.load(file)

In [ ]:
with open("../../data/tacred/json/test.json", "r") as file:
    data_test = json.load(file)

# Deep Learning Models
The code below is partially taken and adapted from the material for the course in Natural Language Processing and Relation Extraction at TU Wien 2023W semester.

In [ ]:
training_data = pd.json_normalize(data_train)
dev_data = pd.json_normalize(data_dev)
test_data = pd.json_normalize(data_test)

In [8]:
training_data = training_data.drop(
    ["docid", "stanford_pos", "stanford_ner", "stanford_head", "stanford_deprel"],
    axis=1,
)
_, training_data["label"] = np.unique(
    training_data["relation"], return_inverse=True
)  # Add a label to encode the different relations

In [9]:
dev_data = dev_data.drop(
    ["docid", "stanford_pos", "stanford_ner", "stanford_head", "stanford_deprel"],
    axis=1,
)
_, dev_data["label"] = np.unique(
    dev_data["relation"], return_inverse=True
)  # Add a label to encode the different relations

In [10]:
test_data = test_data.drop(
    ["stanford_pos", "stanford_ner", "stanford_head", "stanford_deprel"], axis=1
)  # docid retained for analysis later
_, test_data["label"] = np.unique(
    test_data["relation"], return_inverse=True
)  # Add a label to encode the different relations

In [11]:
NUM_CLASSES = len(set(training_data.label))

In [12]:
# Add a new column to store tokenized sentence with subj and obj entities
subj = ["<SUBJ>"]
obj = ["<OBJ>"]

training_data["token_with_entity"] = training_data.apply(
    lambda row: row["token"][: row["subj_start"]]
    + subj
    + row["token"][row["subj_start"] : row["subj_end"] + 1]
    + subj
    + row["token"][row["subj_end"] + 1 :],
    axis=1,
)
training_data["token_with_entity"] = training_data.apply(
    lambda row: row["token_with_entity"][: row["obj_start"]]
    + obj
    + row["token_with_entity"][row["obj_start"] : row["obj_end"] + 1]
    + obj
    + row["token_with_entity"][row["obj_end"] + 1 :],
    axis=1,
)

dev_data["token_with_entity"] = dev_data.apply(
    lambda row: row["token"][: row["subj_start"]]
    + subj
    + row["token"][row["subj_start"] : row["subj_end"] + 1]
    + subj
    + row["token"][row["subj_end"] + 1 :],
    axis=1,
)
dev_data["token_with_entity"] = dev_data.apply(
    lambda row: row["token_with_entity"][: row["obj_start"]]
    + obj
    + row["token_with_entity"][row["obj_start"] : row["obj_end"] + 1]
    + obj
    + row["token_with_entity"][row["obj_end"] + 1 :],
    axis=1,
)

test_data["token_with_entity"] = test_data.apply(
    lambda row: row["token"][: row["subj_start"]]
    + subj
    + row["token"][row["subj_start"] : row["subj_end"] + 1]
    + subj
    + row["token"][row["subj_end"] + 1 :],
    axis=1,
)
test_data["token_with_entity"] = test_data.apply(
    lambda row: row["token_with_entity"][: row["obj_start"]]
    + obj
    + row["token_with_entity"][row["obj_start"] : row["obj_end"] + 1]
    + obj
    + row["token_with_entity"][row["obj_end"] + 1 :],
    axis=1,
)

### Simple NN with 1 hidden layer
Here we use CountVectorize to obtain an **embedding for each sentence**, which is then given as input to the NN.

In [13]:
# Set up torch
seed = 123
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True  # To only use deterministic algorithms
device = "cuda" if torch.cuda.is_available() else "cpu"

In [14]:
# Convert the tokens into a sentence
training_data["text"] = training_data["token"].apply(lambda x: " ".join(x))
dev_data["text"] = dev_data["token"].apply(lambda x: " ".join(x))
test_data["text"] = test_data["token"].apply(lambda x: " ".join(x))


training_data["text_with_entity"] = training_data["token_with_entity"].apply(
    lambda x: " ".join(x)
)
dev_data["text_with_entity"] = dev_data["token_with_entity"].apply(
    lambda x: " ".join(x)
)
test_data["text_with_entity"] = test_data["token_with_entity"].apply(
    lambda x: " ".join(x)
)

Now we work with conversion of **each sentence** into a vector

In [15]:
def prepare_vectorizer(tr_data):
    # max_features: build a vocabulary that only considers the top max_features ordered by term frequency across the corpus.
    vectorizer = CountVectorizer(max_features=3000)

    word_to_ix = vectorizer.fit(tr_data["text"])
    return word_to_ix

In [16]:
word_to_ix = prepare_vectorizer(training_data)
# The vocab size is the length of the vocabulary, or the length of the feature vectors
VOCAB_SIZE = len(word_to_ix.vocabulary_)
print(f"We are working with a vocabulary of size: {VOCAB_SIZE}")

We are working with a vocabulary of size: 3000


In [17]:
tr_data_loader, val_data_loader, test_data_loader = prepare_dataloader(
    training_data, dev_data, test_data, word_to_ix, device
)

In [18]:
# We then define a BATCH_SIZE for our model
BATCH_SIZE = 128

In [19]:
train_iterator, valid_iterator, test_iterator = create_dataloader_iterators(
    tr_data_loader, val_data_loader, test_data_loader, BATCH_SIZE
)

## First DL model
This model consists of a linear layer -> Dropout layer -> ReLU -> linear layer -> softamx output

In [20]:
class BoWDeepClassifier(nn.Module):
    def __init__(self, num_labels, vocab_size, hidden_size):
        super(BoWDeepClassifier, self).__init__()
        # First linear layer
        self.linear1 = nn.Linear(vocab_size, hidden_size)
        # Non-linear activation function between them
        self.relu = torch.nn.ReLU()
        # Second layer
        self.linear2 = nn.Linear(hidden_size, num_labels)
        # Introduce a dropout layer
        self.dropout = nn.Dropout(0.25)

    def forward(self, bow_vec, sequence_lens):
        # Run the input vector through every layer
        output = self.linear1(bow_vec)
        output = self.dropout(output)
        output = self.relu(output)
        output = self.linear2(output)

        # Get the probabilities
        return F.log_softmax(output, dim=1)

In [20]:
INPUT_DIM = VOCAB_SIZE
OUTPUT_DIM = NUM_CLASSES
HIDDEN_SIZE = 200
learning_rate = 0.001

In [21]:
def reinitialize(model):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = torch.nn.NLLLoss()
    model = model.to(device)
    criterion = criterion.to(device)

In [23]:
# Now we try with the basic automatic early-stopping
model = BoWDeepClassifier(OUTPUT_DIM, INPUT_DIM, HIDDEN_SIZE)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

model = model.to(device)
criterion = criterion.to(device)

In [ ]:
training_loop(
    model,
    train_iterator,
    optimizer,
    criterion,
    valid_iterator,
    modelpath="best_models/DL1.pt",
    epoch_number=10,
)

The result are not satisfactory as we are overfitting the training set, with very bad results on the validation set.
This very basic network has many problems:
- First of all the network does not know where the subject and the object are in the sentece.
- As such, two sentences are always classified in the same way, regardless of the subject and object. In our dataset this occurs often.
- We have very unbalanced classes, so we need to rebalance them in some way. Below we include weights to help with this

In [25]:
# We try the same with weights to account for class imbalance
reinitialize(BoWDeepClassifier(OUTPUT_DIM, INPUT_DIM, HIDDEN_SIZE))
model = BoWDeepClassifier(OUTPUT_DIM, INPUT_DIM, HIDDEN_SIZE)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Try a vector of weights inversely proportional to class size (1 / class_size)
cnt_labels = training_data["label"].value_counts()
cnt_labels = cnt_labels.sort_index()
weights = torch.Tensor((1 / cnt_labels))
criterion = nn.NLLLoss(weight=weights)


model = model.to(device)
criterion = criterion.to(device)

In [ ]:
training_loop(
    model,
    train_iterator,
    optimizer,
    criterion,
    valid_iterator,
    modelpath="best_models/DL1_weights.pt",
    epoch_number=10,
)  # Does not change much

### Simple NN with word embedding
For all models from now on we try to encode information about the subject and the object by adding additional special tokens [SUBJ], [OBJ] before and after those.

In [22]:
def prepare_vectorizer_entity(tr_data):
    # max_features: build a vocabulary that only considers the top max_features ordered by term frequency across the corpus.
    vectorizer = CountVectorizer(max_features=3000)

    word_to_ix = vectorizer.fit(tr_data["text_with_entity"])
    return word_to_ix


word_to_ix_entity = prepare_vectorizer_entity(training_data)
# The vocab size is the length of the vocabulary, or the length of the feature vectors
VOCAB_SIZE = len(word_to_ix_entity.vocabulary_)
print(f"We are working with a vocabulary of size: {VOCAB_SIZE}")

We are working with a vocabulary of size: 3000


In [23]:
# Get the analyzer to get the word-id mapping from CountVectorizer
an = (
    word_to_ix_entity.build_analyzer()
)  # handles preprocessing, tokenisation and removal of stop words

# Define a padding value
padding_value = len(word_to_ix.vocabulary_)

In [24]:
(
    tr_data_loader_pad,
    val_data_loader_pad,
    test_data_loader_pad,
) = prepare_dataloader_with_padding(
    training_data, dev_data, test_data, word_to_ix_entity, an, padding_value, device
)

In [25]:
train_iterator_pad, valid_iterator_pad, test_iterator_pad = create_dataloader_iterators(
    tr_data_loader_pad, val_data_loader_pad, test_data_loader_pad, BATCH_SIZE
)

In [26]:
# The INPUT_DIM is the size of our input vectors
INPUT_DIM_PAD = VOCAB_SIZE + 2  # For the padding
OUTPUT_DIM_PAD = NUM_CLASSES
HIDDEN_SIZE_PAD = 20
EMBEDDING_DIM_PAD = 100
learning_rate_PAD = 0.001

In [29]:
class BoWClassifierWithEmbedding(nn.Module):
    def __init__(self, num_labels, vocab_size, embedding_dim):
        super(BoWClassifierWithEmbedding, self).__init__()

        # We define the embedding layer and for this model we train it.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=3001)
        self.embedding.weight.requires_grad = True
        self.linear = nn.Linear(embedding_dim, num_labels)

    def forward(self, text, sequence_lens):
        # First we create the embedded vectors
        embedded = self.embedding(text)
        # We choose max pooling
        pooled = F.max_pool2d(embedded, (embedded.shape[1], 1)).squeeze(
            1
        )  # dim=64x100 (batch_size x embedding_dim)
        return F.log_softmax(self.linear(pooled), dim=1)

In [33]:
model = BoWClassifierWithEmbedding(OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate_PAD)
criterion = nn.NLLLoss()

model = model.to(device)
criterion = criterion.to(device)

In [ ]:
training_loop(
    model,
    train_iterator_pad,
    optimizer,
    criterion,
    valid_iterator_pad,
    modelpath="best_models/DL2.pt",
    epoch_number=10,
)

In [35]:
# Repeat but with weights as above
model = BoWClassifierWithEmbedding(OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate_PAD)
cnt_labels = training_data["label"].value_counts()
cnt_labels = cnt_labels.sort_index()
weights = torch.Tensor((1 / cnt_labels))
criterion = nn.NLLLoss(weight=weights)

model = model.to(device)
criterion = criterion.to(device)

In [ ]:
training_loop(
    model,
    train_iterator_pad,
    optimizer,
    criterion,
    valid_iterator_pad,
    modelpath="best_models/DL2_weights.pt",
    epoch_number=10,
)

### LSTM (unidirectional: accepts Dropout parameter)

In [30]:
class LSTMClassifier_uni(nn.Module):
    def __init__(self, num_labels, vocab_size, embedding_dim, hidden_dim, p_drop=0):
        super(LSTMClassifier_uni, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=3001)
        self.embedding.weight.requires_grad = True

        # Define the LSTM layer
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            num_layers=1,
            bidirectional=False,
        )
        self.linear1 = nn.Linear(hidden_dim, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p_drop)
        self.linear2 = nn.Linear(64, num_labels)

    def forward(self, text, sequence_lens):
        embedded = self.embedding(text)

        # To ensure LSTM doesn't learn gradients for the id of the padding symbol
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, sequence_lens, enforce_sorted=False, batch_first=True
        )
        packed_outputs, (h, c) = self.lstm(packed)
        # extract LSTM outputs (not used here)
        lstm_outputs, lens = nn.utils.rnn.pad_packed_sequence(
            packed_outputs, batch_first=True
        )
        LSTM_output = h[-1]  # last hidden vector from LSTM
        linear_out = self.linear1(LSTM_output)
        linear_out_drop = self.relu(self.dropout(linear_out))
        y = self.linear2(linear_out_drop)
        log_probs = F.log_softmax(y, dim=1)
        return log_probs

In [ ]:
model = LSTMClassifier_uni(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD
)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

model = model.to(device)

criterion = nn.NLLLoss().to(device)
training_loop(
    model,
    train_iterator_pad,
    optimizer,
    criterion,
    valid_iterator_pad,
    modelpath="best_models/LSTM1_nodrop.pt",
    epoch_number=10,
)

In [ ]:
model = LSTMClassifier_uni(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD
)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

model = model.to(device)
weights = torch.Tensor((1 / cnt_labels))
criterion = nn.NLLLoss(weight=weights).to(device)
training_loop(
    model,
    train_iterator_pad,
    optimizer,
    criterion,
    valid_iterator_pad,
    modelpath="best_models/LSTM1_nodrop_weights.pt",
    epoch_number=10,
)

In [ ]:
model = LSTMClassifier_uni(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, p_drop=0.1
)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

model = model.to(device)

criterion = nn.NLLLoss().to(device)
training_loop(
    model,
    train_iterator_pad,
    optimizer,
    criterion,
    valid_iterator_pad,
    modelpath="best_models/LSTM1_Drop10.pt",
    epoch_number=10,
)

In [ ]:
model = LSTMClassifier_uni(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, p_drop=0.25
)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

model = model.to(device)

criterion = nn.NLLLoss().to(device)
training_loop(
    model,
    train_iterator_pad,
    optimizer,
    criterion,
    valid_iterator_pad,
    modelpath="best_models/LSTM1_Drop25.pt",
    epoch_number=10,
)

In [ ]:
model = LSTMClassifier_uni(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, p_drop=0.50
)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

model = model.to(device)

criterion = nn.NLLLoss().to(device)
training_loop(
    model,
    train_iterator_pad,
    optimizer,
    criterion,
    valid_iterator_pad,
    modelpath="best_models/LSTM1_Drop50.pt",
    epoch_number=10,
)

### LSTM2 (Bi-directional, accepts dropout parameter)

In [27]:
class LSTMClassifier_bi(nn.Module):
    def __init__(self, num_labels, vocab_size, embedding_dim, hidden_dim, p_drop=0):
        super(LSTMClassifier_bi, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=3001)
        self.embedding.weight.requires_grad = True

        # Define the LSTM layer
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            num_layers=1,
            bidirectional=True,
        )
        self.linear1 = nn.Linear(2 * hidden_dim, 64)
        # Dropout to overcome overfitting
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p_drop)
        self.linear2 = nn.Linear(64, num_labels)

    def forward(self, text, sequence_lens):
        embedded = self.embedding(text)

        # To ensure LSTM doesn't learn gradients for the id of the padding symbol
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, sequence_lens, enforce_sorted=False, batch_first=True
        )
        packed_outputs, (h, c) = self.lstm(packed)
        y = self.linear1(
            torch.concat((h[0], h[1]), dim=1)
        )  # Concatenate the final hidden vectors for the fw and bw pass.
        y = self.relu(self.dropout(y))
        y = self.linear2(y)
        log_probs = F.log_softmax(y, dim=1)
        return log_probs

In [ ]:
model = LSTMClassifier_bi(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, p_drop=0
)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

model = model.to(device)

criterion = nn.NLLLoss().to(device)
training_loop(
    model,
    train_iterator_pad,
    optimizer,
    criterion,
    valid_iterator_pad,
    modelpath="best_models/LSTM2_nodrop.pt",
    epoch_number=10,
)

In [ ]:
model = LSTMClassifier_bi(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, p_drop=0.1
)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

model = model.to(device)

criterion = nn.NLLLoss().to(device)
training_loop(
    model,
    train_iterator_pad,
    optimizer,
    criterion,
    valid_iterator_pad,
    modelpath="best_models/LSTM2_Drop10.pt",
    epoch_number=10,
)

In [ ]:
model = LSTMClassifier_bi(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, p_drop=0.25
)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

model = model.to(device)

criterion = nn.NLLLoss().to(device)
training_loop(
    model,
    train_iterator_pad,
    optimizer,
    criterion,
    valid_iterator_pad,
    modelpath="best_models/LSTM2_Drop25.pt",
    epoch_number=10,
)

In [ ]:
model = LSTMClassifier_bi(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, p_drop=0.50
)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

model = model.to(device)

criterion = nn.NLLLoss().to(device)
training_loop(
    model,
    train_iterator_pad,
    optimizer,
    criterion,
    valid_iterator_pad,
    modelpath="best_models/LSTM2_Drop50.pt",
    epoch_number=10,
)

## Model results
Below the F1 score obtained as described is reported for the different models.\
The best model is chosen by looking at this measure on the validation set. Afterwards, error analysis will be performed on the test set.

In [31]:
# Obtain metrics of this model on dev set
DL1 = BoWDeepClassifier(OUTPUT_DIM, INPUT_DIM, HIDDEN_SIZE).to(device)
DL1.load_state_dict(torch.load("best_models/DL1.pt"))
DL1.eval()
# Obtain results on validation set
print(f"Results for DL1")
labels, predictions = extract_predictions(DL1, valid_iterator)
_ = score(labels, predictions)

# Obtain metrics of this model on dev set
DL1_weights = BoWDeepClassifier(OUTPUT_DIM, INPUT_DIM, HIDDEN_SIZE).to(device)
DL1_weights.load_state_dict(torch.load("best_models/DL1_weights.pt"))
DL1_weights.eval()
# Obtain results on validation set
print(f"Results for DL1 with weights")
labels, predictions = extract_predictions(DL1_weights, valid_iterator)
_ = score(labels, predictions)

# Obtain metrics of this model on dev set
DL2 = BoWClassifierWithEmbedding(OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD).to(
    device
)
DL2.load_state_dict(torch.load("best_models/DL2.pt"))
DL2.eval()
# Obtain results on validation set
print(f"Results for DL2")
labels, predictions = extract_predictions(DL2, valid_iterator_pad)
_ = score(labels, predictions)

# Obtain metrics of this model on dev set
DL2_weights = BoWClassifierWithEmbedding(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD
).to(device)
DL2_weights.load_state_dict(torch.load("best_models/DL2_weights.pt"))
DL2_weights.eval()
# Obtain results on validation set
print(f"Results for DL2 with weights")
labels, predictions = extract_predictions(DL2_weights, valid_iterator_pad)
_ = score(labels, predictions)

# Obtain metrics of this model on dev set
LSTM1 = LSTMClassifier_uni(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD
).to(device)
LSTM1.load_state_dict(torch.load("best_models/LSTM1_nodrop.pt"))
LSTM1.eval()
# Obtain results on validation set
print(f"Results for LSTM1 without dropout")
labels, predictions = extract_predictions(LSTM1, valid_iterator_pad)
_ = score(labels, predictions)

# Obtain metrics of this model on dev set
LSTM1_weights = LSTMClassifier_uni(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD
).to(device)
LSTM1_weights.load_state_dict(torch.load("best_models/LSTM1_nodrop_weights.pt"))
LSTM1_weights.eval()
# Obtain results on validation set
print(f"Results for LSTM1 without dropout and with weights")
labels, predictions = extract_predictions(LSTM1_weights, valid_iterator_pad)
_ = score(labels, predictions)


# Obtain metrics of this model on dev set
LSTM1_Drop10 = LSTMClassifier_uni(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, 0.1
).to(device)
LSTM1_Drop10.load_state_dict(torch.load("best_models/LSTM1_Drop10.pt"))
LSTM1_Drop10.eval()
# Obtain results on validation set
print(f"Results for LSTM1 with Dropout 0.1")
labels, predictions = extract_predictions(LSTM1_Drop10, valid_iterator_pad)
_ = score(labels, predictions)

# Obtain metrics of this model on dev set
LSTM1_Drop25 = LSTMClassifier_uni(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, 0.25
).to(device)
LSTM1_Drop25.load_state_dict(torch.load("best_models/LSTM1_Drop25.pt"))
LSTM1_Drop25.eval()
# Obtain results on validation set
print(f"Results for LSTM1 with Dropout 0.25")
labels, predictions = extract_predictions(LSTM1_Drop25, valid_iterator_pad)
_ = score(labels, predictions)

# Obtain metrics of this model on dev set
LSTM1_Drop50 = LSTMClassifier_uni(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, 0.50
).to(device)
LSTM1_Drop50.load_state_dict(torch.load("best_models/LSTM1_Drop50.pt"))
LSTM1_Drop50.eval()
# Obtain results on validation set
print(f"Results for LSTM1 with Dropout 0.50")
labels, predictions = extract_predictions(LSTM1_Drop50, valid_iterator_pad)
_ = score(labels, predictions)


# Obtain metrics of this model on dev set
LSTM2_nodrop = LSTMClassifier_bi(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, 0.1
).to(device)
LSTM2_nodrop.load_state_dict(torch.load("best_models/LSTM2_nodrop.pt"))
LSTM2_nodrop.eval()
# Obtain results on validation set
print(f"Results for LSTM2 without dropout")
labels, predictions = extract_predictions(LSTM2_nodrop, valid_iterator_pad)
_ = score(labels, predictions)

# Obtain metrics of this model on dev set
LSTM2_Drop10 = LSTMClassifier_bi(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, 0.1
).to(device)
LSTM2_Drop10.load_state_dict(torch.load("best_models/LSTM2_Drop10.pt"))
LSTM2_Drop10.eval()
# Obtain results on validation set
print(f"Results for LSTM2 with Dropout 0.1")
labels, predictions = extract_predictions(LSTM2_Drop10, valid_iterator_pad)
_ = score(labels, predictions)

# Obtain metrics of this model on dev set
LSTM2_Drop25 = LSTMClassifier_bi(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, 0.25
).to(device)
LSTM2_Drop25.load_state_dict(torch.load("best_models/LSTM2_Drop25.pt"))
LSTM2_Drop25.eval()
# Obtain results on validation set
print(f"Results for LSTM2 with Dropout 0.25")
labels, predictions = extract_predictions(LSTM2_Drop25, valid_iterator_pad)
_ = score(labels, predictions)

# Obtain metrics of this model on dev set
LSTM2_Drop50 = LSTMClassifier_bi(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, 0.50
).to(device)
LSTM2_Drop50.load_state_dict(torch.load("best_models/LSTM2_Drop50.pt"))
LSTM2_Drop50.eval()
# Obtain results on validation set
print(f"Results for LSTM2 with Dropout 0.50")
labels, predictions = extract_predictions(LSTM2_Drop50, valid_iterator_pad)
_ = score(labels, predictions)

Results for DL1
Precision (micro): 17.694%
   Recall (micro): 1.214%
       F1 (micro): 2.272%
Results for DL1 with weights
Precision (micro): 7.417%
   Recall (micro): 25.294%
       F1 (micro): 11.470%
Results for DL2
Precision (micro): 24.593%
   Recall (micro): 2.226%
       F1 (micro): 4.082%
Results for DL2 with weights
Precision (micro): 4.470%
   Recall (micro): 18.396%
       F1 (micro): 7.193%
Results for LSTM1 without dropout
Precision (micro): 27.221%
   Recall (micro): 6.990%
       F1 (micro): 11.124%
Results for LSTM1 without dropout and with weights
Precision (micro): 2.151%
   Recall (micro): 8.756%
       F1 (micro): 3.453%
Results for LSTM1 with Dropout 0.1
Precision (micro): 34.312%
   Recall (micro): 9.032%
       F1 (micro): 14.300%
Results for LSTM1 with Dropout 0.25
Precision (micro): 25.996%
   Recall (micro): 7.800%
       F1 (micro): 11.999%
Results for LSTM1 with Dropout 0.50
Precision (micro): 39.170%
   Recall (micro): 7.119%
       F1 (micro): 12.049%
Res

## Error analysis
For error analysis only the best model is considered. According to the results above we can see that this appears to be the bi-directional model with dropout probability of 0.25 (LSTM2_Dropout25). Further discussion of the results is presented in the associated report.\
Even though DL techniques are generally considered black-box, we can try and obtain some insights from our model. Additional difficulties in performing more detailed error analysis come from the large number of data and relations.
### Results on test dataset
For starters we consider the performance of the model on the test dataset, using the same metric as before.

In [28]:
LSTM2_Drop25 = LSTMClassifier_bi(
    OUTPUT_DIM_PAD, INPUT_DIM_PAD, EMBEDDING_DIM_PAD, HIDDEN_SIZE_PAD, 0.25
).to(device)
LSTM2_Drop25.load_state_dict(torch.load("best_models/LSTM2_Drop25.pt"))
LSTM2_Drop25.eval()
# Obtain results on test set
print(f"Results for LSTM2 with Dropout 0.25")
labels, predictions = extract_predictions(LSTM2_Drop25, test_iterator_pad)
_ = score(labels, predictions)

Results for LSTM2 with Dropout 0.25
Precision (micro): 34.156%
   Recall (micro): 10.827%
       F1 (micro): 16.442%


These results appear to be more or less in line with those obtained on the validation dataset. However, there is some slight decrease in precision.

### Do our tags help
First of all we try and analyze whether the specified SUBJ and OBJ tags, help the model in distinguishing the same sentence when the relation to be classified is between different elements of it. This was one of the major shortcomings that were noted in the first model (DL1), particularly in the case without weights.\
Consider for example the test dataset entries associated with `docid` *eng-NG-31-142760-10093410*, where we have in some cases the same sentence repeated but with different subject and object.

In [33]:
# First of all convert numeric vectors to strings for simplicity
labels = [int(l) for l in labels]
labels_dict = dict(
    [(lab, rel) for lab, rel in zip(test_data["label"], test_data["relation"])]
)
labels_dict = dict(sorted(labels_dict.items()))
labels_str = [labels_dict[l] for l in labels]
preds_str = [labels_dict[p] for p in predictions]

In [35]:
# Append the predicted relation to the dataset
test_data["predicted_relation"] = preds_str

In [36]:
test_data_docid = test_data.copy()[test_data["docid"] == "eng-NG-31-142760-10093410"]
test_data_docid.sort_values("token")[5:10]

,id,docid,relation,token,subj_start,subj_end,obj_start,obj_end,subj_type,obj_type,label,token_with_entity,text,text_with_entity,predicted_relation
3849,098f6fb926d1dcbfc761,eng-NG-31-142760-10093410,per:title,"[He, shook, my, hand, and, kissed, me, on, bot...",12,13,17,17,PERSON,TITLE,41,"[He, shook, my, hand, and, kissed, me, on, bot...",He shook my hand and kissed me on both cheeks ...,He shook my hand and kissed me on both cheeks ...,per:title
11149,098f665fb9578c5aaac1,eng-NG-31-142760-10093410,no_relation,"[He, shook, my, hand, and, kissed, me, on, bot...",12,13,21,24,PERSON,DURATION,0,"[He, shook, my, hand, and, kissed, me, on, bot...",He shook my hand and kissed me on both cheeks ...,He shook my hand and kissed me on both cheeks ...,no_relation
6099,098f6f09754dca578187,eng-NG-31-142760-10093410,per:religion,"[He, shook, my, hand, and, kissed, me, on, bot...",12,13,26,26,PERSON,RELIGION,34,"[He, shook, my, hand, and, kissed, me, on, bot...",He shook my hand and kissed me on both cheeks ...,He shook my hand and kissed me on both cheeks ...,no_relation
7600,098f60af8f6a687b6b35,eng-NG-31-142760-10093410,no_relation,"[He, shook, my, hand, and, kissed, me, on, bot...",12,13,29,29,PERSON,LOCATION,0,"[He, shook, my, hand, and, kissed, me, on, bot...",He shook my hand and kissed me on both cheeks ...,He shook my hand and kissed me on both cheeks ...,no_relation
8393,098f665fb9f56219e1bf,eng-NG-31-142760-10093410,no_relation,"[He, shook, my, hand, and, kissed, me, on, bot...",12,13,0,0,PERSON,PERSON,0,"[<OBJ>, He, <OBJ>, shook, my, hand, and, kisse...",He shook my hand and kissed me on both cheeks ...,<OBJ> He <OBJ> shook my hand and kissed me on ...,no_relation


We can see that while the model is far from working near state of the art techniques, it is capable of assigning different labels to the same sentence when the specified subject and object (using tags <SUBJ> and <OBJ>).\
It correctly assigns label 41 (corresponding to relation `per:title`) to one of the relations as we can see above.

### More detailed analysis of the performance on different 

In [37]:
len(set(preds_str))  # Only predicts 9 different relations

9

We already see some issues in our model as it only assigns to the different relations 9 of the available ones. This is clearly quite unsatisfactory and below we include some additional details, with precision and recall for each category.

In [38]:
_ = score_str(labels_str, preds_str, verbose=True)

Per-relation statistics:
org:alternate_names                  P:  39.58%  R:  35.68%  F1:  37.53%  #: 213
org:city_of_headquarters             P: 100.00%  R:   0.00%  F1:   0.00%  #: 82
org:country_of_headquarters          P: 100.00%  R:   0.00%  F1:   0.00%  #: 108
org:dissolved                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 2
org:founded                          P: 100.00%  R:   0.00%  F1:   0.00%  #: 37
org:founded_by                       P: 100.00%  R:   0.00%  F1:   0.00%  #: 68
org:member_of                        P: 100.00%  R:   0.00%  F1:   0.00%  #: 18
org:members                          P: 100.00%  R:   0.00%  F1:   0.00%  #: 31
org:number_of_employees/members      P: 100.00%  R:   0.00%  F1:   0.00%  #: 19
org:parents                          P: 100.00%  R:   0.00%  F1:   0.00%  #: 62
org:political/religious_affiliation  P: 100.00%  R:   0.00%  F1:   0.00%  #: 10
org:shareholders                     P: 100.00%  R:   0.00%  F1:   0.00%  #: 13
org:stateorpro

We can note that we have a Recall of 0% on many of the instances. In fact, as already seen above our model only predicts some of the total 42 categories on the sentences of the test dataset.\
Moreover, we can note that we have a recall value different than 0 only for the 6 relations reported below.
- per:title
- per:date_of_birth 
- per:age
- org:website
- org:top_members/employees
- org:alternate_names

These results are reasonable on some of these relations, as they contain a large number of examples of on the training set, however, some of the ones with a non-zero recall only contain a few. Similarly, some relations with a large number of examples on the training set are also misclassified.\
Examples of these (as can be seen from the counts in the training dataset) are relation `per:employee_of` or `org:country_of_headquarters` where both have a not too small number of instances in the training set. Let us look into more detail in how they are classified in the test set.

In [39]:
training_data["relation"].value_counts()[0:10]

no_relation                    51367
per:title                       2287
org:top_members/employees       1753
per:employee_of                 1441
org:alternate_names              742
org:country_of_headquarters      438
per:countries_of_residence       410
per:age                          364
org:city_of_headquarters         357
per:cities_of_residence          351
Name: relation, dtype: int64

In [42]:
# Let us see how instances of per:employee_of are classified
test_data_per_employee = test_data.copy()[test_data["relation"] == "per:employee_of"]
test_data_per_employee["predicted_relation"].value_counts()

no_relation                  217
org:top_members/employees     25
per:title                     18
org:alternate_names            4
Name: predicted_relation, dtype: int64

It appears that in most cases the sentences are wrongly classified as `no_relation`. This may be due to our model suffering from the large number of no relation instances.\
Some other instances are instead classified mainly as `org:top_members/employees` or `per:title`, which are not completely unrelated, and both of them appear more often than `per:employee_of` in the training set.

In [43]:
# Let us see how instances of org:country_of_headquarters are classified
test_data_per_employee = test_data.copy()[
    test_data["relation"] == "org:country_of_headquarters"
]
test_data_per_employee["predicted_relation"].value_counts()

no_relation                  97
org:top_members/employees     6
org:alternate_names           5
Name: predicted_relation, dtype: int64

Also for this relation we note that the main issue seems to be that most relations are classified once more into `no_relation`.
Further discussion of these results is provided in the report.